# 2. Data Understanding

## 2.1 Objective & Strategy
The goal is to understand the CICIDS2017 dataset (approx. 6 million network flows). 
Given the size of the dataset (803 MB in Parquet), loading the entire dataset into memory for EDA is inefficient. We will:
1. Read a representative sample to inspect data types and distributions.
2. Check for data quality issues (`NaN`, `inf`).
3. Analyze the target variable (`Label`) to quantify the class imbalance.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# 1. Cibler UNIQUEMENT le dossier MachineLearningCVE (standard ML)
csv_pattern = "../data/raw/MachineLearningCSV/MachineLearningCVE/*.csv"
csv_files = glob.glob(csv_pattern)

# Fallback si le chemin est différent
if not csv_files:
    csv_pattern = "../data/raw/**/*.csv"
    csv_files = glob.glob(csv_pattern, recursive=True)
    csv_files = [f for f in csv_files if "WorkingHours" in f] # Filtrer les fichiers

print(f"Nombre de fichiers CSV trouvés : {len(csv_files)}")

if len(csv_files) == 0:
    print("⚠️ ERREUR : Aucun fichier CSV trouvé.")
else:
    print(f"Chargement d'un échantillon (100 000 lignes max) depuis : {os.path.basename(csv_files[0])}")
    
    # Lire les 100 000 premières lignes
    df_sample = pd.read_csv(csv_files[0], nrows=100000)
    
    # Nettoyage
    df_sample.columns = df_sample.columns.str.strip()
    df_sample = df_sample.loc[:, ~df_sample.columns.duplicated()]

    print(f"Taille de l'échantillon : {df_sample.shape[0]} lignes, {df_sample.shape[1]} colonnes")

## 2.2 Target Variable Distribution

In cybersecurity, the target variable is highly imbalanced. Let's quantify this imbalance to justify our choice of metrics (PR-AUC and Asymmetric Cost).

In [ ]:
# Attention à l'espace dans le nom de la colonne d'origine, on l'a nettoyé normalement
target_col = 'Label'

print("--- Target Distribution (Absolute Counts) ---")
print(df_sample[target_col].value_counts())

print("\n--- Target Distribution (Percentages) ---")
print((df_sample[target_col].value_counts(normalize=True) * 100).round(2))

## 2.3 Data Quality Issues

Network flow features often contain infinite (`inf`) values due to division by zero (e.g., a flow duration of 0). We must identify them before preprocessing.

In [ ]:
# 1. Vérification des valeurs manquantes (NaN)
missing = df_sample.isnull().sum()
print("--- Colonnes avec valeurs manquantes (NaN) ---")
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("Aucune valeur manquante.")

# 2. Vérification des valeurs infinies (Inf)
numeric_cols = df_sample.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df_sample[numeric_cols]).sum()
inf_counts = inf_counts[inf_counts > 0]

print("\n--- Colonnes avec valeurs infinies (Inf) ---")
if len(inf_counts) > 0:
    print(inf_counts)
else:
    print("Aucune valeur infinie.")